In [192]:
import pandas as pd
import numpy as np
import os
import calendar
from datetime import datetime
np.seterr(divide='ignore',invalid='ignore')
#台站数据整理
#风速 平均气温 日降水量 日照数 相对湿度 最低气温 最高气温
DataNameDict={'R20':'日降水量','Tmax':'最高气温','Tmin':'最低气温','T':'平均气温',
            'S':'日照数','U':'相对湿度','WS':'风速'}

def d_to_jd(time):
    fmt = '%Y-%m-%d'
    dt = datetime.strptime(time, fmt)
    tt = dt.timetuple()
    return tt.tm_year * 1000 + tt.tm_yday

def jd_to_time(time):
    dt = datetime.strptime(time, '%Y%j').date()
    fmt ='%Y-%m-%d'
    return dt.strftime(fmt)
#判断是否闰年
def isLeapYear(year):
    if not year%4 and  year%100 or not year%400:
        return 1
    return 0

workpath=r'f:\08SPEI'
datapath=os.path.join(workpath,r'01_data\climate\climate_daiy_19610101_20230601')

data_merge_path=os.path.join(datapath,'合并')
data_Interpolate_path=os.path.join(datapath,'缺测插补')
                      
timeintervals=['1961-1965','1966-1970','1971-1975','1976-1980',
              '1981-1985','1986-1990','1991-1995','1996-2000',
               '2001-2005','2006-2010','2011-2015','2016-2021','2022-2023']
startyear,endyear=1961,2022
stinfo=pd.read_excel(os.path.join(datapath,'站点编号及信息.xlsx'))
st_sum=len(stinfo['station'])

datakeys=['R20','Tmax','Tmin','T','S','U','WS']


In [20]:
'''''''''''''''''''''''''''''
#step1 逐日数据 合并
'''''''''''''''''''''''''''''
fnpath=datapath
dfs=pd.DataFrame()
print("正在读取数据....")
for sel_t in timeintervals:   
    #print(sel_t)
    startyr=int(sel_t.split('-')[0])
    endyr=int(sel_t.split('-')[1])
    f=str(startyr)+'-01-01_'+str(endyr)+'-12-31_day_R20_Tmax_Tmin_T_S_U_WS.txt'   
    df=pd.read_csv(os.path.join(fnpath,f),sep='\t')

    df['doys']=df['time'].map(d_to_jd) 
    df['time']=pd.to_datetime(df['time'])
    df['year']=df['time'].dt.year
    df['doys']=df['doys']-df['year']*1000    
    dfs=pd.concat([dfs,df])
dfs[dfs[datakeys]>999]=np.nan      
dfs[dfs[datakeys]<-999]=np.nan  

dfs=dfs[dfs['year']<=endyear]
#输出 
#分站点所以要素一起保存
print("正在输出数据....")
outpath=data_merge_path
if not os.path.exists(outpath): os.makedirs(outpath)
for i in range(st_sum): 
    st_no=stinfo['station'][i]
    outdf=pd.DataFrame()
    for iy in range(startyear,endyear+1):  
        yeardays=365+isLeapYear(iy)
        df1=pd.DataFrame(np.arange(1,yeardays+1).reshape((yeardays,1)),columns=['doys'])         
        sel_df=dfs[(dfs['stationId']==st_no)&(dfs['year']==iy)][['doys']+datakeys]
        df1=pd.merge(df1,sel_df,how='left',on='doys')       
        df1['time']=df1['doys']+iy*1000
        df1['time']=df1['time'].astype(str)
        df1['time']=df1['time'].map(jd_to_time) 
        outdf=pd.concat([outdf,df1])
    outfn=os.path.join(outpath,str(st_no)+".csv")
    #print(i,st_sum,outfn)
    outdf.to_csv(outfn)         # 路径和文
print(datetime.now())         

正在读取数据....
正在输出数据....
2023-06-04 14:14:12.692938


In [23]:
'''''''''''''''''''''''''''''
step2:缺测统计---空值插补前
'''''''''''''''''''''''''''''
fnpath=data_merge_path
row_list=[]
print("统计空值插补前缺测统计")
for i in range(st_sum): 
                    
    st_no=stinfo['station'][i]
    fn=os.path.join(fnpath,str(st_no)+".csv")
    df=pd.read_csv(fn)
    df['time']=pd.to_datetime(df['time'])
    df['year']=df['time'].dt.year
    for iy in range(startyear,endyear+1):
        for sel_d in datakeys:
            nansum=df[(df['year']==iy)][sel_d].isnull().sum()
            if nansum>0:  
                #print(st_no,iy,sel_d,nansum)
                row_list.append([st_no,iy,sel_d,nansum])

outfn=os.path.join(data_Interpolate_path,'缺测统计_插补前.csv')
outdf = pd.DataFrame(row_list,columns=['station','year','data','Nansum'])
outdf.to_csv(outfn)

#统计插补后数据缺测次数iy=1970

fn=os.path.join(data_Interpolate_path,'缺测统计_插补前.csv')
df0 = pd.read_csv(fn)
iy=1969
df=df0
df=df[df['year']>=iy]
print(iy,len(df['station'].unique()),df['station'].unique())

统计空值插补前缺测统计
1969 125 [56444 56483 56489 56497 56533 56543 56548 56567 56582 56585 56586 56594
 56595 56596 56641 56643 56645 56646 56649 56651 56652 56654 56664 56669
 56673 56684 56688 56697 56739 56742 56745 56746 56748 56751 56752 56755
 56756 56757 56761 56763 56764 56766 56767 56768 56772 56774 56775 56777
 56778 56781 56782 56783 56785 56786 56788 56790 56835 56836 56838 56839
 56840 56841 56842 56843 56844 56846 56849 56851 56854 56856 56862 56863
 56867 56869 56870 56871 56872 56873 56875 56876 56878 56879 56880 56881
 56882 56883 56885 56886 56889 56891 56898 56944 56946 56948 56949 56950
 56951 56952 56954 56958 56959 56961 56962 56964 56966 56969 56970 56973
 56975 56976 56977 56978 56982 56984 56985 56986 56987 56989 56991 56992
 56994 56995 56996 59007 59205]


In [193]:
'''
step3:缺测空值插补
首先采用前后两天均有观测值时，空值用前后两天观测值求平均进行插值
而连续缺测10天以内的数据采用气候值进行插补，连续缺测10天以上站点进行剔除。
考虑到年际间风速变化相较降水和气温的波动较小，对缺测35天以上站点进行剔除。
'''
fnpath=data_merge_path
outpath=data_Interpolate_path

for i in range(st_sum): 
    st_no=stinfo['station'][i]
    fn=os.path.join(fnpath,str(st_no)+".csv")
    df=pd.read_csv(fn).reset_index()
    for sel_d in datakeys:  
        datalist=df[sel_d].values.tolist()
        index_list=np.where(np.isnan(datalist))[0] 
        #连续空值索引值及连续个数:vlist,wlist
        vlist,wlist=[],[]
        tlist = [index_list[k+1]-index_list[k] for k in range(len(index_list)-1)] #range后还可以加if条件筛选
        tlist.append(0)
        k=0
        while k<(len(tlist)):
            v=tlist[k]
            w=1
            while v==1:
                v=tlist[k+w]
                w=w+1   
            #print(index_list[k],w)
            vlist.append(index_list[k])
            wlist.append(w)
            k=k+w
        
        #而连续缺测10天以内的数据采用气候值进行插补，连续缺测10天以上站点进行剔除。                      
        threshold=15
        if sel_d=='WS': threshold=35    
        idx_max=len(datalist)
        for j in range(len(vlist)):               
            idx=vlist[j]
            w=wlist[j]                      
            if (w==1)&(idx>0) & (idx<=idx_max):                
                datalist[idx]=np.nanmean(datalist[idx-1:idx+2])#前后两天数据插补     
                #print(idx,datalist[idx-1:idx+2])
            if (w<threshold) & (w>1):
                for iw in range(w): 
                    d0=df.loc[idx]['doys']
                    datalist[idx]=np.nanmean(df[df['doys']==d0][sel_d])#用研究时段内气候平均值数据插补 
                    #print('climate',w,idx,datalist[idx])                     
                    idx=idx+1
        #保存插值后的结果
        df[sel_d]=datalist   
    outfn=os.path.join(outpath,str(st_no)+".csv")
    #print(i,'/',st_sum,'   ',st_no,outfn)
    df.to_csv(outfn)
print('缺测空值插补 OK!',datetime.now())    



缺测空值插补 OK! 2023-06-04 18:08:49.011006


In [194]:
'''
step4:对插补后数据集进行缺测统计
'''
fnpath=data_Interpolate_path
row_list=[]
print('正在处理.....')    
for i in range(st_sum): 
    st_no=stinfo['station'][i]
    fn=os.path.join(fnpath,str(st_no)+".csv")
    df=pd.read_csv(fn)
    df['time']=pd.to_datetime(df['time'])
    df['year']=df['time'].dt.year
    for iy in range(startyear,endyear+1):
        for sel_d in datakeys:
            nansum=df[(df['year']==iy)][sel_d].isnull().sum()
            if nansum>0:  
                #print(st_no,iy,sel_d,nansum)
                row_list.append([st_no,iy,sel_d,nansum])

outfn=os.path.join(data_Interpolate_path,'缺测统计_插补后.csv')
outdf = pd.DataFrame(row_list,columns=['station','year','data','Nansum'])
outdf.to_csv(outfn)
print('This OK!',datetime.now())     

正在处理.....
This OK! 2023-06-04 18:24:53.030348


In [195]:
'''''''''''''''''''''''''''''
step4:统计插补后数据缺测次数iy=1970
'''''''''''''''''''''''''''''
fn=os.path.join(data_Interpolate_path,'缺测统计_插补前.csv')
df0 = pd.read_csv(fn)
iy=1970
df=df0
df=df[df['year']>=iy]
print(iy,len(df['station'].unique()),df['station'].unique())

fn=os.path.join(data_Interpolate_path,'缺测统计_插补后.csv')
df0 = pd.read_csv(fn)
iy=1969
df=df0
df=df[(df['year']>=iy)&(df['year']<2023)]
print(iy,len(df['station'].unique()),df['station'].unique())

print(datetime.now())       

1970 125 [56444 56483 56489 56497 56533 56543 56548 56567 56582 56585 56586 56594
 56595 56596 56641 56643 56645 56646 56649 56651 56652 56654 56664 56669
 56673 56684 56688 56697 56739 56742 56745 56746 56748 56751 56752 56755
 56756 56757 56761 56763 56764 56766 56767 56768 56772 56774 56775 56777
 56778 56781 56782 56783 56785 56786 56788 56790 56835 56836 56838 56839
 56840 56841 56842 56843 56844 56846 56849 56851 56854 56856 56862 56863
 56867 56869 56870 56871 56872 56873 56875 56876 56878 56879 56880 56881
 56882 56883 56885 56886 56889 56891 56898 56944 56946 56948 56949 56950
 56951 56952 56954 56958 56959 56961 56962 56964 56966 56969 56970 56973
 56975 56976 56977 56978 56982 56984 56985 56986 56987 56989 56991 56992
 56994 56995 56996 59007 59205]
1969 10 [56641 56643 56645 56646 56649 56673 56783 56882 56989 59205]
2023-06-04 18:24:53.095330


In [196]:
##step 05 对研究时段和选定的站点数据进行检验，是否有缺
fnpath=data_Interpolate_path
datakeys=['R20','Tmax','Tmin','T','S','U','WS']

iy=1969
df=pd.DataFrame(data=[56641, 56643, 56645, 56646, 56649 ,56673, 56783, 56882, 56989, 59205],columns=['station'])
sel_stinfo=pd.concat([stinfo,df], ignore_index=True, verify_integrity=True,sort=True) 
sel_stinfo.drop_duplicates(subset=['station'],keep=False,inplace=True) 
sel_stinfo.to_excel(os.path.join(fnpath,'站点编号及信息_选定_1969-'+str(endyear)+'.xlsx'))
'''
i=1
for st_no in sel_stinfo['station']:  
    fn=os.path.join(fnpath,str(st_no)+".csv")
    #print(i,st_no,iy,fn)
    df=pd.read_csv(fn)
    
    df['time']=pd.to_datetime(df['time'])
    df['year']=df['time'].dt.year      
    df1=df[df['year']>=iy]
    for sel_d in datakeys:
        lostsum = df1[sel_d].isnull().sum()    
        if lostsum>0 : print(i,st_no,sel_d,lostsum)
    i=i+1
'''
print(fnpath,'站点编号及信息_选定_1969-2022.xlsx',datetime.now())   

f:\08SPEI\01_data\climate\climate_daiy_19610101_20230601\缺测插补 站点编号及信息_选定_1969-2022.xlsx 2023-06-04 18:25:23.560658
